### this notebook is to test the functions in part 2 of the coursework

In [5]:
import pandas as pd

In [7]:
df = pd.read_csv("/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw-pack-2026/texts/hansard10000.csv")
display(df.head(2))

,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson


In [8]:
# rename the labours(Co-op) in party column to labour
df["party"] = df["party"].replace("Labour (Co-op)", "Labour")
display(df.head(2))

,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson


In [18]:
# remove any rows where the value of the ‘party’ column is not one of the four
# most common party names, and remove the ‘Speaker’ value.
parties_in_df = df["party"].unique()
# print(parties_in_df)

print(df["party"].value_counts())
df_cleaned = df[df["party"].isin(["Labour", "Conservative", "Scottish National Party", "Liberal Democrat"])]

# remove the speakername column
df_cleaned = df_cleaned.drop(columns=["speakername"])
display(df_cleaned.head(2))
print(len(df_cleaned))

# remove any rows where the value in the ‘speech class’ column is not ‘Speech’.
df_cleaned = df_cleaned[df_cleaned["speech_class"] == "Speech"]
print(len(df_cleaned))

# remove any rows where the text in the ‘speech’ column is less than 1000 characters long.
df_cleaned = df_cleaned[df_cleaned["speech"].str.len() >= 1000]
print(len(df_cleaned))
print(df_cleaned.shape)

party
Conservative                        6192
Labour                              2108
Scottish National Party              560
Liberal Democrat                     221
Speaker                              208
Democratic Unionist Party            140
Independent                           58
Plaid Cymru                           51
Social Democratic & Labour Party      21
Green Party                           17
Alliance                              12
Alba Party                             1
Name: count, dtype: int64


,speech,party,constituency,date,speech_class,major_heading,year
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020


9081
9081
2112
(2112, 7)


### the following is to test the code for question 2b

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score

In [22]:
# !pip install scikit-learn

In [24]:
# question 2b
# vectorise using TfidfVectorizer
vectorizer = TfidfVectorizer(stop_words="english", max_features=3000)
X = vectorizer.fit_transform(df_cleaned["speech"])
y = df_cleaned["party"]

In [25]:
# train test split with stratifed sampling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=26, stratify=y
)

In [32]:
# train a Random Forest classifier
rf = RandomForestClassifier(random_state=26, n_estimators=300)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print("macro F1-score:", f1_score(y_test, rf_pred, average="macro"))
print("Random Forest Classification Report:")
print(classification_report(y_test, rf_pred))

macro F1-score: 0.42190741964275363
Random Forest Classification Report:
                         precision    recall  f1-score   support

           Conservative       0.70      0.97      0.82       250
                 Labour       0.76      0.42      0.54       125
       Liberal Democrat       0.00      0.00      0.00        15
Scottish National Party       0.78      0.21      0.33        33

               accuracy                           0.71       423
              macro avg       0.56      0.40      0.42       423
           weighted avg       0.70      0.71      0.67       423



/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defin

In [30]:
# train a SVM with linear kernel
svm = LinearSVC(random_state=26)
svm.fit(X_train, y_train)
svm_pred = svm.predict(X_test)
print("macro F1-score:", f1_score(y_test, svm_pred, average="macro"))
print("SVM (Linear) Classification Report:")
print(classification_report(y_test, svm_pred))

macro F1-score: 0.5257393582301793
SVM (Linear) Classification Report:
                         precision    recall  f1-score   support

           Conservative       0.76      0.90      0.83       250
                 Labour       0.67      0.60      0.63       125
       Liberal Democrat       1.00      0.07      0.12        15
Scottish National Party       0.76      0.39      0.52        33

               accuracy                           0.74       423
              macro avg       0.80      0.49      0.53       423
           weighted avg       0.74      0.74      0.72       423

